In [22]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

import SCRIPTS.jsonDownloader as jd
from SCRIPTS.redditLinkRetriever import fetch_saved_post_links, save_links_txt
from SCRIPTS.mediaDownloader import download_embedded_media
from SCRIPTS.mediaOrganizer import organize_downloads
from SCRIPTS.redgifDownloader import process_external
from SCRIPTS.cloudflareUploader import upload_media
from SCRIPTS.r2_audit import audit_local_vs_r2

In [23]:
# https://old.reddit.com/prefs/apps
# Set to True to avoid making any changes
DRY_RUN_MEDIA = False
DRY_RUN_ORGANIZE = False

DRY_RUN_CLOUDFLARE = False   # True = preview only, no upload
ACCOUNT_INDEX = 2            # choose which R2 account (1, 2, ...)
CHECK_ONLY = False           # True = just check existence, no upload

DRY_RUN_FINAL = False

In [24]:
# Retrieve saved post links for the specified user
links = fetch_saved_post_links()

In [25]:
len(links)

138

In [26]:
links[:5]

['https://www.reddit.com/r/ABGHeavens/comments/1ogiwta/alabama_abg_slut',
 'https://www.reddit.com/r/bangmybully/comments/1ogx4wo/maybe_its_time_to_change_profession',
 'https://www.reddit.com/r/ABGHeavens/comments/1ogmcw3/the_best_abg_ive_seen',
 'https://www.reddit.com/r/ABGHeavens/comments/1ofz5d3/since_a_lot_of_you_liked_delilah_heres_a_video',
 'https://www.reddit.com/r/tipofmypenis/comments/1lwdqfx/who_is_this_abg_getting_fucked_on_the_beach']

# NEW POST VALIDATION

This section validates new posts from reddits saved folder

In [27]:
csv_path = Path("ordered_posts.csv")
raw_df = pd.read_csv(csv_path)

POST_ID_RE = re.compile(r"/comments/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
SHORT_RE   = re.compile(r"redd\.it/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
max_order_num = raw_df.order_num.max()

def strip_trailing_slash(url: str) -> str:
    # remove trailing slashes only at the very end (doesn't touch scheme)
    return url.rstrip("/")

def extract_post_id(url: str) -> str | None:
    """
    Try to extract a post id from:
      - standard permalink: .../comments/<postid>/...
      - shortlink: https://redd.it/<postid>
    """
    m = POST_ID_RE.search(url)
    if m:
        return m.group(1)
    m = SHORT_RE.search(url)
    if m:
        return m.group(1)
    return None

existing_ids = set(str(x).lower() for x in raw_df.get("post_id", pd.Series([])).dropna())

new_rows = []
next_order = max_order_num + 1
seen_in_batch = set()  # avoid duplicates within this run

for raw_link in reversed(links):
    link = strip_trailing_slash(raw_link)
    post_id = extract_post_id(link)
    if not post_id:
        continue
    pid = post_id.lower()

    # Only add if NOT already in CSV and not already queued this batch
    if pid in existing_ids or pid in seen_in_batch:
        continue

    new_rows.append({
        "order_num": next_order,
        "link": link,
        "post_id": post_id,
        "date_added": datetime.utcnow().isoformat(timespec="seconds"),
    })
    seen_in_batch.add(pid)
    next_order += 1

# Preview as a DataFrame
new_df = pd.DataFrame(new_rows)
new_df


C:\Users\minds\AppData\Local\Temp\ipykernel_11552\787629229.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_added": datetime.utcnow().isoformat(timespec="seconds"),


,order_num,link,post_id,date_added
0,1341,https://www.reddit.com/r/ABGHeavens/comments/1...,1ogmcw3,2025-10-27T09:59:46
1,1342,https://www.reddit.com/r/bangmybully/comments/...,1ogx4wo,2025-10-27T09:59:46
2,1343,https://www.reddit.com/r/ABGHeavens/comments/1...,1ogiwta,2025-10-27T09:59:46


In [28]:
final_df = pd.concat([raw_df, new_df], ignore_index=True)
final_df = final_df.sort_values(by="order_num", ascending=False).reset_index(drop=True)

In [29]:
import importlib
importlib.reload(jd)

jd.configure(
    DATA_ROOT="Out",
    SKIP_EXISTING=False,
    REPORTS_DIR="__reports"
    )

summary = jd.process_all(new_df["link"].tolist(), show_progress=True)
summary

  0%|          | 0/3 [00:00<?, ?post/s]

Done. Success: 0, Skipped: 3, Failed: 0


{'success': 0, 'skipped': 3, 'failed': 0}

# MEDIA DOWNLOADER
Reviews the external and media json folders in **Out/**, downloading:
- Images
- Gifs
- Videos

In [30]:
folders = ["external", "media"]
download_stats = []

# point to your inputs/outputs explicitly
for mediaType in folders:
    download_stats.append(download_embedded_media(
        media_json_dir=Path("Out/" + mediaType),   # where your *.json live
        media_out_dir=Path("Media/media_files"),  # where downloads should go
        write_fail_csv_to=Path("__reports/media_report" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
        show_progress=True,
    ))

download_stats

[{'downloaded': 0,
  'failed': 1,
  'skipped': 0,
  'fail_rows': [{'id': '1ogx4wo', 'reason': 'no_reddit_media_url'}],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/external')},
 {'downloaded': 11,
  'failed': 0,
  'skipped': 0,
  'fail_rows': [],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/media')}]

In [31]:
move_stats = organize_downloads(
    input_dir="Media/media_files",  # where your downloader wrote files
    output_dir="Media",             # where Images/, Videos/, Gifs/ live
    strategy="move",
    conflict="move_existing",
    show_progress=True,
    dry_run=DRY_RUN_ORGANIZE,       # set True to preview
    prune_empty_galleries=True,     # remove empty src folders after moving
)

move_stats

Organizing media:   0%|          | 0/11 [00:00<?, ?file/s]

[SKIPPED→AlreadyMoved] 1ogmcw3.jpeg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogmcw3.jpeg
[SKIPPED→AlreadyMoved] 01.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\01.jpg
[SKIPPED→AlreadyMoved] 02.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\02.jpg
[SKIPPED→AlreadyMoved] 03.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\03.jpg
[SKIPPED→AlreadyMoved] 04.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\04.jpg
[SKIPPED→AlreadyMoved] 05.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\05.jpg
[SKIPPED→AlreadyMoved] 06.jpg → S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\AlreadyMoved\media_files\1ogiwta\06.jpg
[SKIPPED→AlreadyMoved] 07.jpg → S:\mi

{'moved': 0,
 'copied': 0,
 'linked': 0,
 'skipped': 11,
 'unknown': 0,
 'dry_run': False,
 'strategy': 'move',
 'conflict': 'move_existing',
 'input_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files',
 'output_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media',
 'errors': [],
 'created_dirs': set(),
 'pruned_dirs': ['S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1ogiwta']}

# REDGIF DOWNLOADER

Downloads redgifs from external json folder in **Out/**

In [32]:
stats = process_external(
    media_json_dir=Path("Out/external"),
    media_out_dir=Path("Media/RedGiphys"),
    write_fail_csv_to=Path("__reports/redgif_report_" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    write_links_csv_to=Path("__reports/external_links" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    show_progress=True,
    dry_run=DRY_RUN_MEDIA,
    overwrite_downloads=False,
)

stats

Found 1 external post JSONs in Out\external


Scanning external posts:   0%|          | 0/1 [00:00<?, ?post/s]

Saved external links to: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\__reports\external_links20251027-025948.csv


{'external_rows': [{'id': '1ogx4wo',
   'link': 'https://www.redgifs.com/watch/blackandwhitesomealbatross',
   'domain': 'www.redgifs.com'}],
 'redgifs_failed': [],
 'links_csv_path': WindowsPath('__reports/external_links20251027-025948.csv'),
 'fail_csv_path': None,
 'out_dir': WindowsPath('Media/RedGiphys')}

# CLOUDFLARE VERIFICATION & UPLOAD

In [33]:
raw_output = []
uploadsData = []

for mediaType in ["Images", "Videos", "Gifs", "RedGiphys"]:
    try:
        result = upload_media(
            input_path=Path("Media") / mediaType,  # where local files/galleries live
            r2_prefix=mediaType,                   # must match bucket prefix
            account_idx=ACCOUNT_INDEX - 1,         # <— choose which R2 credentials to use
            dry_run=DRY_RUN_CLOUDFLARE,            # preview vs. real upload
            overwrite=False,                       # don't overwrite existing objects
            check_only=CHECK_ONLY,                 # <— enable to just check existence
        )

        raw_output.append(result)
        # choose which list you want to visualize depending on mode
        if CHECK_ONLY:
            uploadsData.extend(result["exists"] + result["missing"])
        else:
            uploadsData.extend(result["planned"])

    except Exception as e:
        print(f"⚠️ Error on {mediaType}: {e}")
        continue

⚠️ Error on Videos: Input path not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos
⚠️ Error on Gifs: Input path not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Gifs


In [34]:
results_df = pd.DataFrame(uploadsData)
pd.set_option('display.max_rows', None)
results_df

,local,r2_key,bytes,content_type,status,account_index,bucket
0,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogmcw3.jpeg,144572,image/jpeg,planned,1,media-archive
1,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/01.jpg,99721,image/jpeg,planned,1,media-archive
2,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/02.jpg,85138,image/jpeg,planned,1,media-archive
3,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/03.jpg,89046,image/jpeg,planned,1,media-archive
4,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/04.jpg,128334,image/jpeg,planned,1,media-archive
5,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/05.jpg,121866,image/jpeg,planned,1,media-archive
6,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/06.jpg,117252,image/jpeg,planned,1,media-archive
7,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/07.jpg,218124,image/jpeg,planned,1,media-archive
8,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/08.jpg,250512,image/jpeg,planned,1,media-archive
9,S:\minds\Desktop\Downloader and Reddit System\...,Images/1ogiwta/09.jpg,236985,image/jpeg,planned,1,media-archive


# VERIFY UPLOAD

In [35]:
cats = ["Images", "RedGiphys", "Gifs", "Videos"]
all_rows = []

for cat in cats:
    try:
        res = audit_local_vs_r2(
            local_root=Path("Media") / cat,  # e.g., Media/Images
            r2_prefixes=[cat],               # matches your bucket key prefix
            account_indices=None,            # all accounts
            write_csv_to=None,               # (optional) per-cat CSV
            show_progress=True,
        )
        rows = res["rows"]
        for r in rows:
            r["category"] = cat
        all_rows.extend(rows)
    except Exception as e:
        print(f"⚠️ Error auditing {cat}: {e}")
        continue

audit_results = pd.DataFrame(all_rows)
pd.set_option("display.max_rows", None)
audit_results

Auditing: 100%|██████████| 2/2 [00:00<?, ?file/s]

⚠️ Error auditing Gifs: Local images root not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Gifs
⚠️ Error auditing Videos: Local images root not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos


,local_rel,local_ext,all_expected_keys,matched,match_type,matched_prefix,matched_key,remote_ext,same_ext,matched_account_index,matched_bucket,note,category
0,1ogmcw3.jpeg,.jpeg,Images/1ogmcw3.jpeg,True,exact,Images,Images/1ogmcw3.jpeg,.jpeg,True,1,media-archive,exact_match,Images
1,1ogiwta/01.jpg,.jpg,Images/1ogiwta/01.jpg,True,exact,Images,Images/1ogiwta/01.jpg,.jpg,True,1,media-archive,exact_match,Images
2,1ogiwta/02.jpg,.jpg,Images/1ogiwta/02.jpg,True,exact,Images,Images/1ogiwta/02.jpg,.jpg,True,1,media-archive,exact_match,Images
3,1ogiwta/03.jpg,.jpg,Images/1ogiwta/03.jpg,True,exact,Images,Images/1ogiwta/03.jpg,.jpg,True,1,media-archive,exact_match,Images
4,1ogiwta/04.jpg,.jpg,Images/1ogiwta/04.jpg,True,exact,Images,Images/1ogiwta/04.jpg,.jpg,True,1,media-archive,exact_match,Images
5,1ogiwta/05.jpg,.jpg,Images/1ogiwta/05.jpg,True,exact,Images,Images/1ogiwta/05.jpg,.jpg,True,1,media-archive,exact_match,Images
6,1ogiwta/06.jpg,.jpg,Images/1ogiwta/06.jpg,True,exact,Images,Images/1ogiwta/06.jpg,.jpg,True,1,media-archive,exact_match,Images
7,1ogiwta/07.jpg,.jpg,Images/1ogiwta/07.jpg,True,exact,Images,Images/1ogiwta/07.jpg,.jpg,True,1,media-archive,exact_match,Images
8,1ogiwta/08.jpg,.jpg,Images/1ogiwta/08.jpg,True,exact,Images,Images/1ogiwta/08.jpg,.jpg,True,1,media-archive,exact_match,Images
9,1ogiwta/09.jpg,.jpg,Images/1ogiwta/09.jpg,True,exact,Images,Images/1ogiwta/09.jpg,.jpg,True,1,media-archive,exact_match,Images


In [41]:
if DRY_RUN_FINAL or (False in audit_results["matched"].value_counts().keys()):
    print(final_df.head(20))
    print("Dry run enabled; ordered_posts not changed")
else:
    print("Updating ordered_posts.csv with new posts...")
    final_df.to_csv("ordered_posts.csv", index=False)

Updating ordered_posts.csv with new posts...
